# Exercises XP — Prompt Engineering

## Solution académique complète

Ce notebook traite les six exercices de prompt engineering autour de :

- l’analyse des prompts vagues ;
- le role prompting ;
- les instructions précises ;
- les contraintes de ton, longueur et format ;
- les prompts multi-parties ;
- l’ajout de contexte ;
- le choix du style de prompt ;
- la validation de contraintes ;
- la réduction des hallucinations.

Les prompts finaux sont en anglais lorsque le livrable attendu est en
anglais. Les explications méthodologiques sont en français.

## Objectifs pédagogiques

À la fin du notebook, vous saurez :

1. diagnostiquer les faiblesses d’un prompt vague ;
2. définir un rôle utile pour le modèle ;
3. transformer une demande générale en instructions exécutables ;
4. préciser l’audience, le but et la source ;
5. contrôler la structure, le ton et la longueur ;
6. choisir entre un style exploratoire, structuré, conversationnel ou
   fonctionnel ;
7. tester automatiquement certaines contraintes ;
8. réduire les hallucinations avec des règles de grounding.

## Cadre général : R-T-C-F-Q-V

| Élément | Question |
|---|---|
| **R — Role** | Quel rôle le modèle doit-il adopter ? |
| **T — Task** | Quelle action exacte doit-il exécuter ? |
| **C — Context** | Quelles informations sont nécessaires ? |
| **F — Format** | Quelle structure de sortie est attendue ? |
| **Q — Quality constraints** | Quelles limites de ton, longueur ou contenu ? |
| **V — Verification** | Comment éviter omissions et hallucinations ? |

Template général :

```text
Act as [ROLE].

Your task is to [TASK].

Context:
[CONTEXT]

Output format:
[FORMAT]

Constraints:
[QUALITY CONSTRAINTS]

Before answering:
[VERIFICATION RULES]
```

# Exercise 1 — Rewrite and Optimize a Vague Prompt

Prompt initial :

> “Write something about productivity tips.”

Le but réel est de promouvoir l’application de concentration **FlowNest**
dans une publication LinkedIn.

## 1.1 Trois problèmes critiques

In [ ]:
critical_issues = [
    (
        "No target audience is defined. The model does not know whether "
        "the post is for remote workers, managers, students, founders, or "
        "another LinkedIn audience."
    ),
    (
        "The objective and brand positioning are missing. The prompt does "
        "not mention FlowNest, its value proposition, or the desired call "
        "to action."
    ),
    (
        "There are no platform, tone, structure, or length constraints. "
        "The model may produce an essay, generic advice, or content that "
        "does not encourage LinkedIn engagement."
    ),
]

for index, issue in enumerate(critical_issues, start=1):
    print(f"{index}. {issue}")

Les problèmes correspondent aux dimensions demandées :

- **brand voice** : FlowNest et son positionnement sont absents ;
- **target users** : l’audience LinkedIn n’est pas définie ;
- **platform engagement** : aucune structure, accroche ou CTA n’est exigé.

## 1.2 Prompt amélioré

In [ ]:
rewritten_prompt = """
Act as a LinkedIn content strategist for FlowNest, a focus and productivity
app designed for busy professionals, remote workers, and startup teams.

Write one LinkedIn post that promotes FlowNest through practical
productivity advice.

Requirements:
1. Start with a short hook that reflects a common concentration problem.
2. Give exactly 3 actionable productivity tips.
3. Connect at least one tip naturally to FlowNest's ability to support
   focused work sessions and reduce distractions.
4. End with one clear call to action inviting readers to try FlowNest.
5. Use a friendly, credible, and professional brand voice.
6. Use short paragraphs or bullet points suitable for LinkedIn.
7. Keep the post between 120 and 170 words.
8. Add no more than 3 relevant hashtags.
9. Do not invent product features that are not stated in this prompt.

Use this structure whenever we promote productivity tools on social media,
adapting the audience, product benefits, and platform constraints as needed.
""".strip()

print(rewritten_prompt)

## 1.3 Techniques utilisées

- **Role prompting** : LinkedIn content strategist.
- **Instruction prompting** : exactement trois conseils, une accroche et un
  appel à l’action.
- **Format, tone and length constraints** : ton professionnel, 120–170 mots,
  trois hashtags maximum.
- **Réutilisabilité** : la dernière phrase transforme le prompt en template.

# Exercise 2 — Multi-Part Prompt for Quiz Generation

Une enseignante de sciences de 7e année fournit un article sur les
éruptions volcaniques. Le modèle doit produire un résumé, un quiz et une
structure prête pour Google Slides.

## 2.1 Prompt complet

In [ ]:
quiz_prompt = """
Act as an experienced 7th-grade science teacher and educational content
designer.

Use only the article provided between <article> and </article> to create a
Google Slides-ready mini lesson for students aged 11 to 13.

<article>
[PASTE THE VOLCANIC ERUPTIONS ARTICLE HERE]
</article>

PART 1 — SIMPLE SUMMARY
- Write exactly 2 bullet points.
- Each bullet must contain one main idea from the article.
- Use simple vocabulary and no more than 20 words per bullet.

PART 2 — MULTIPLE-CHOICE QUIZ
- Create exactly 3 questions.
- Each question must be answerable from the article.
- Give exactly 3 options labelled A, B, and C.
- Include exactly 1 correct answer and 2 plausible distractors.
- Avoid trick questions, double negatives, and advanced vocabulary.

PART 3 — ANSWER KEY
- List each question number and its correct letter.
- Add a one-sentence explanation based only on the article.

GOOGLE SLIDES FORMAT
Slide 1: Title — "Volcanic Eruptions: Key Ideas"
Slide 2: Two-point summary
Slide 3: Question 1
Slide 4: Question 2
Slide 5: Question 3
Slide 6: Answer key

STYLE AND QUALITY RULES
- Use a friendly, encouraging classroom tone.
- Keep slide text short and easy to scan.
- Do not add scientific facts absent from the article.
- If the article lacks enough information for 3 valid questions, state
  what is missing instead of inventing content.
- Return only the slide-ready content.
""".strip()

print(quiz_prompt)

## 2.2 Trois améliorations par rapport au prompt vague

In [ ]:
improvements = [
    (
        "The learners are precisely defined as students aged 11 to 13, "
        "which controls vocabulary, tone, and difficulty."
    ),
    (
        "The complete deliverable is specified: 2 summary bullets, "
        "3 questions, 3 options per question, and an answer key."
    ),
    (
        "The prompt imposes a Google Slides structure and grounding rules, "
        "preventing unsupported facts and irrelevant formatting."
    ),
]

for index, improvement in enumerate(improvements, start=1):
    print(f"{index}. {improvement}")

Le prompt évite les mauvaises sorties grâce à :

1. une audience précise plutôt que le terme vague « kids » ;
2. une décomposition en trois parties mesurables ;
3. une obligation d’utiliser uniquement l’article et un format Slides.

# Exercise 3 — Add Context, Get Better Results

Prompt initial :

> “Summarize this report.”

Le résumé doit servir à une présentation financière mensuelle de trois
minutes.

## 3.1 Tableau des contextes manquants

In [ ]:
import pandas as pd

context_table = {
    "Role": {
        "missing": "Yes",
        "add": (
            "Ask the model to act as a financial analyst who can identify "
            "material business movements."
        ),
    },
    "Audience": {
        "missing": "Yes",
        "add": (
            "Specify a non-technical executive team that needs concise, "
            "decision-relevant information."
        ),
    },
    "Purpose": {
        "missing": "Yes",
        "add": (
            "State that the summary supports a 3-minute monthly finance "
            "update and must highlight risks and decisions."
        ),
    },
    "Input Source": {
        "missing": "Yes",
        "add": (
            "Delimit the report and require the model to use only figures "
            "and statements contained in it."
        ),
    },
    "Format/Style": {
        "missing": "Yes",
        "add": (
            "Request a headline, 3 takeaways with data, risks, and one "
            "closing recommendation."
        ),
    },
    "Constraints": {
        "missing": "Yes",
        "add": (
            "Limit the script to about 300–360 words, preserve currencies "
            "and periods, and flag unavailable comparisons."
        ),
    },
}

context_table_df = (
    pd.DataFrame.from_dict(context_table, orient="index")
    .reset_index()
    .rename(
        columns={
            "index": "Context Type",
            "missing": "Is it Missing?",
            "add": "What Should Be Added?",
        }
    )
)

context_table_df

## 3.2 Prompt contextuel amélioré

In [ ]:
finance_prompt = """
Act as a senior financial analyst preparing an executive briefing.

Audience:
The summary is for a non-technical executive leadership team.

Purpose:
Create a spoken summary for a 3-minute monthly finance update. The audience
needs to understand performance, major changes, risks, and decisions.

Source:
Use only the report between <finance_report> and </finance_report>.
Do not infer missing figures or use external financial information.

<finance_report>
[PASTE THE MONTHLY FINANCE REPORT HERE]
</finance_report>

Output format:
1. Executive headline — one sentence describing the overall direction.
2. Three key takeaways — each must include:
   - the business point;
   - at least one supporting figure;
   - the comparison period, target, or variance when available;
   - why the point matters.
3. Risks or concerns — maximum 2 bullets.
4. Recommended next action — one concise sentence.

Constraints:
- Use clear, non-technical executive language.
- Preserve exact currencies, percentages, units, and reporting periods.
- Keep the full output between 300 and 360 words.
- Prioritize revenue, margin, operating costs, cash, and material variances.
- If a requested figure is absent, write "Not provided in the report".
- Do not add generic advice unrelated to the report.
""".strip()

print(finance_prompt)

Ce prompt contient les six types de contexte : rôle, audience, objectif,
source, format et contraintes.

# Exercise 4 — Match Prompt to Purpose

## Style choisi : Structured

Cas d’usage : chatbot e-commerce chargé de répondre à une demande de statut
de commande.

In [ ]:
chosen_style = "Structured"

prompt_template = """
Act as a customer support assistant for an e-commerce company.

Help the customer understand an order status using only the order
information supplied by the support system.

Customer message:
<customer_message>
{customer_message}
</customer_message>

Verified order data:
<order_data>
{order_data}
</order_data>

Follow this response structure exactly:

1. Acknowledgement
- Greet the customer briefly.
- Acknowledge the specific concern.

2. Verified status
- State the current order status.
- Mention the latest verified date, location, or delivery estimate.
- Never invent tracking events.

3. Next step
- Explain what the customer should do next.
- Give only options allowed by the supplied company policy.
- If data is missing, ask one focused follow-up question.

4. Escalation
- Escalate when the order is lost, marked delivered but not received,
  involves a payment dispute, or falls outside the supplied policy.

5. Closing
- End with a short, reassuring sentence.

Rules:
- Use a calm, respectful, helpful tone.
- Keep the response under 140 words.
- Do not expose internal notes or private account data.
- Do not promise refunds, credits, or dates unless supported.
- Return only the customer-facing message.
""".strip()

key_style_features = [
    (
        "A fixed five-part sequence controls acknowledgement, status, next "
        "step, escalation, and closing."
    ),
    (
        "Conditional rules define how missing data, delays, disputes, and "
        "escalations must be handled consistently."
    ),
]

justification = """
A structured style is best because order-support requests require consistent
checks, predictable escalation, and compliance with policy. An exploratory
style would generate unnecessary options. A conversational style could omit
required checks. A purely functional transformation would be too narrow for
this multi-step support process.
""".strip()

print("Chosen style:", chosen_style)
print("\nPROMPT\n", prompt_template)
print("\nKEY FEATURES")
for feature in key_style_features:
    print("-", feature)
print("\nJUSTIFICATION\n", justification)

Les deux caractéristiques du style structuré sont :

- une séquence de réponse fixe ;
- des règles conditionnelles et d’escalade.

Ce style est préférable car la conformité et la cohérence comptent davantage
que la créativité.

# Exercise 5 — Control Style, Structure, and Length

Contraintes pour le blurb PulseOne Mini :

- ton amical ;
- puces ;
- maximum 50 mots ;
- battery life ;
- fitness tracking ;
- Bluetooth compatibility.

## 5.1 Première version du prompt

In [ ]:
pulseone_prompt_v1 = """
Act as an email copywriter for a wearable technology brand.

Write a short product blurb for the PulseOne Mini smartwatch.

Requirements:
- Use a friendly and engaging tone.
- Use bullet points.
- Keep the complete blurb under 50 words.
- Mention battery life, fitness tracking, and Bluetooth compatibility.
- Return only the blurb.
""".strip()

print(pulseone_prompt_v1)

## 5.2 Exemple de sortie à évaluer

Dans une utilisation réelle, remplacez cette valeur par la réponse obtenue
après exécution du prompt dans ChatGPT.

In [ ]:
pulseone_output_v1 = """
- Meet PulseOne Mini, your friendly everyday smartwatch.
- Enjoy reliable battery life and simple fitness tracking for daily goals.
- Stay connected with Bluetooth compatibility across your favorite devices.
""".strip()

print(pulseone_output_v1)

## 5.3 Validation automatique

In [ ]:
import re

def count_words(text: str) -> int:
    return len(
        re.findall(
            r"\b[\w]+(?:['’\-][\w]+)*\b",
            text,
            flags=re.UNICODE,
        )
    )

def extract_non_empty_lines(text: str):
    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

def is_bullet_line(line: str) -> bool:
    return bool(re.match(r"^[-*•]\s+\S+", line))

def evaluate_pulseone_output(text: str) -> dict:
    normalized = text.lower()
    lines = extract_non_empty_lines(text)

    feature_checks = {
        "battery_life": "battery" in normalized,
        "fitness_tracking": (
            "fitness" in normalized
            and ("tracking" in normalized or "track" in normalized)
        ),
        "bluetooth_compatibility": (
            "bluetooth" in normalized
            and ("compatib" in normalized or "connect" in normalized)
        ),
    }

    friendly_markers = [
        "friendly",
        "enjoy",
        "meet",
        "easy",
        "simple",
        "stay connected",
        "your",
    ]

    return {
        "word_count": count_words(text),
        "word_count_ok": count_words(text) <= 50,
        "bullet_format_ok": (
            bool(lines)
            and all(is_bullet_line(line) for line in lines)
        ),
        "exactly_three_bullets": len(lines) == 3,
        "features": feature_checks,
        "features_covered": all(feature_checks.values()),
        "tone_ok": any(
            marker in normalized
            for marker in friendly_markers
        ),
    }

eval_v1 = evaluate_pulseone_output(pulseone_output_v1)
eval_v1

Le nombre de mots, le format et les fonctionnalités peuvent être contrôlés
automatiquement. Le ton reste partiellement subjectif et nécessite une
validation humaine.

## 5.4 Prompt renforcé

In [ ]:
pulseone_prompt_v2 = """
Act as a concise email copywriter for a friendly wearable-tech brand.

Create the final email blurb for the PulseOne Mini smartwatch.

NON-NEGOTIABLE OUTPUT RULES:
1. Write exactly 3 bullet points.
2. Do not exceed 50 words in total, including all bullet points.
3. Mention "battery life" in one bullet.
4. Mention "fitness tracking" in one bullet.
5. Mention "Bluetooth compatibility" in one bullet.
6. Use a friendly and engaging tone.
7. Do not add a heading, introduction, conclusion, disclaimer, or hashtags.
8. Do not invent technical specifications, durations, health claims, or
   device brands.

Before returning the answer, silently verify:
- exactly 3 bullet points;
- 50 words or fewer;
- all 3 required feature phrases are present.

Return only the 3 bullet points.
""".strip()

print(pulseone_prompt_v2)

La version renforcée utilise des règles non négociables, un nombre exact de
puces, les expressions obligatoires et une étape de vérification silencieuse.

# Exercise 6 — Hallucination Spotting and Mitigation

Affirmation non soutenue ajoutée par le modèle :

> “Over 50% of marine species are projected to go extinct by 2050.”

Cette phrase n’apparaît pas dans l’article.

## 6.1 Prompt anti-hallucination

In [ ]:
no_hallucination_prompt = """
Act as a scientific summarization assistant.

Summarize the peer-reviewed article provided between <article> and
</article>.

<article>
[PASTE THE ARTICLE ON CLIMATE CHANGE AND MARINE BIODIVERSITY HERE]
</article>

Rules:
- Use only information contained in the article.
- Do not add external facts, estimates, dates, percentages, examples, or
  conclusions.
- Do not infer claims that the authors do not explicitly make.
- Preserve the authors' level of certainty.
- Distinguish observed findings from projections or hypotheses.
- Write one concise paragraph followed by 3 key findings.
""".strip()

print(no_hallucination_prompt)

## 6.2 Version renforcée avec vérification

In [ ]:
verification_prompt = """
Act as a careful scientific evidence reviewer.

Use only the text between <article> and </article> to produce a grounded
summary.

<article>
[PASTE THE PEER-REVIEWED ARTICLE HERE]
</article>

VERIFICATION PROCEDURE:
1. Identify explicitly stated findings, figures, dates, and limitations.
2. Include a claim only when directly supported by the article.
3. Preserve qualifiers such as "may", "could", "suggests", "associated
   with", or "projected".
4. Never convert an association into causation.
5. Never create a statistic, extinction estimate, timeline, or opinion.
6. If a requested detail is absent, write:
   "This information is not available in the provided article."
7. Verify every factual sentence against the article and remove any
   unsupported sentence.

OUTPUT:
- Evidence-based summary: 120–160 words.
- Key findings: exactly 3 bullets.
- Limitations reported by the authors: maximum 2 bullets.
- Unsupported-information note: list important unavailable information.

Return no external knowledge and no uncited assumptions.
""".strip()

print(verification_prompt)

## 6.3 Deux stratégies de mitigation

In [ ]:
mitigation_strategies = [
    {
        "strategy": "Source grounding and explicit exclusion",
        "explanation": (
            "The model is restricted to the delimited article and is "
            "forbidden from adding external statistics, dates, estimates, "
            "or conclusions."
        ),
    },
    {
        "strategy": "Claim verification and uncertainty handling",
        "explanation": (
            "The model must verify every factual sentence, preserve "
            "qualifiers, and state that information is unavailable instead "
            "of filling gaps."
        ),
    },
]

for item in mitigation_strategies:
    print(item["strategy"])
    print(item["explanation"])
    print()

## 6.4 Domaines professionnels à haut risque

In [ ]:
risky_domains = {
    "Healthcare": {
        "why": (
            "An invented diagnosis, dosage, contraindication, or treatment "
            "recommendation can cause physical harm or delay proper care."
        )
    },
    "Legal services": {
        "why": (
            "Fabricated laws, deadlines, precedents, or duties can cause "
            "lost rights, financial liability, or invalid decisions."
        )
    },
}

for domain_name, details in risky_domains.items():
    print(f"{domain_name}: {details['why']}")

Un prompt anti-hallucination ne suffit pas toujours. Dans les domaines
sensibles, il faut également contrôler les sources, appliquer les droits
d’accès, journaliser les réponses et prévoir une validation humaine.

# Synthèse des techniques

| Technique | Utilité | Exemple |
|---|---|---|
| Role prompting | Orienter l’expertise | LinkedIn strategist |
| Audience context | Adapter vocabulaire et priorités | 11–13-year-olds |
| Instruction prompting | Définir les actions | 3 tips, 3 MCQs |
| Format constraints | Rendre la sortie exploitable | Google Slides |
| Length constraints | Contrôler la densité | 50 words |
| Tone specification | Contrôler la voix | Friendly, professional |
| Delimiters | Séparer la source | `<article>...</article>` |
| Verification language | Réduire les hallucinations | Verify every claim |
| Fallback behavior | Gérer l’absence d’information | Not provided |
| Programmatic validation | Tester les règles | Word-count checker |

# Checklist finale

1. Le rôle est-il utile et précis ?
2. La tâche contient-elle un verbe d’action clair ?
3. L’audience est-elle définie ?
4. La source est-elle identifiée et délimitée ?
5. Le format de sortie est-il explicite ?
6. Les contraintes sont-elles mesurables ?
7. Le ton correspond-il au canal ?
8. Le modèle sait-il quoi faire si une information manque ?
9. Les hypothèses sont-elles interdites lorsque nécessaire ?
10. La sortie peut-elle être validée automatiquement ou humainement ?

# Conclusion

Un prompt de qualité ne consiste pas seulement à ajouter plus de mots. Il
faut ajouter les bonnes informations :

```text
Rôle + tâche + contexte + format + contraintes + vérification
```

Les exercices montrent que :

- un prompt vague oblige le modèle à deviner ;
- le rôle ne remplace pas les instructions ;
- une tâche complexe doit être décomposée ;
- l’audience détermine le vocabulaire ;
- les contraintes objectives doivent être contrôlées ;
- la structure doit correspondre à la finalité ;
- les hallucinations sont réduites par le grounding et la vérification ;
- une validation humaine reste indispensable dans les domaines sensibles.